<a href="https://colab.research.google.com/github/fv050795/modelagem-de-dados-/blob/main/exemplo-tabelas-programa-banco-de-dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import sqlite3
import datetime

def criar_e_preencher_banco_de_dados(nome_banco='loja.db'):
    """
    Cria as tabelas do banco de dados e preenche com dados de exemplo.
    """
    conn = None  # Initialize conn to None
    try:
        conn = sqlite3.connect(nome_banco)
        conn.row_factory = sqlite3.Row  # Permite acessar os dados por nome da coluna
        cursor = conn.cursor()

        # 1. Tabela CLIENTE
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Cliente (
                id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
                nome TEXT NOT NULL,
                email TEXT UNIQUE NOT NULL,
                telefone TEXT,
                endereco TEXT
            );
        """)
        print("Tabela 'Cliente' criada com sucesso.")

        # 2. Tabela VENDEDOR
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Vendedor (
                id_vendedor INTEGER PRIMARY KEY AUTOINCREMENT,
                nome TEXT NOT NULL,
                cpf TEXT UNIQUE NOT NULL,
                comissao REAL DEFAULT 0.0
            );
        """)
        print("Tabela 'Vendedor' criada com sucesso.")

        # 3. Tabela PRODUTO
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Produto (
                id_produto INTEGER PRIMARY KEY AUTOINCREMENT,
                nome TEXT NOT NULL,
                descricao TEXT,
                preco REAL NOT NULL,
                estoque INTEGER NOT NULL
            );
        """)
        print("Tabela 'Produto' criada com sucesso.")

        # 4. Tabela PEDIDOS
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Pedidos (
                id_pedido INTEGER PRIMARY KEY AUTOINCREMENT,
                data_pedido TEXT NOT NULL,
                valor_total REAL NOT NULL,
                id_cliente_fk INTEGER NOT NULL,
                id_vendedor_fk INTEGER NOT NULL,
                status_pedido TEXT DEFAULT 'Pendente',
                FOREIGN KEY (id_cliente_fk) REFERENCES Cliente(id_cliente),
                FOREIGN KEY (id_vendedor_fk) REFERENCES Vendedor(id_vendedor)
            );
        """)
        print("Tabela 'Pedidos' criada com sucesso.")

        # 5. Tabela CONTEM (Associativa entre Pedidos e Produto)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Contem (
                id_pedido_fk INTEGER NOT NULL,
                id_produto_fk INTEGER NOT NULL,
                quantidade INTEGER NOT NULL,
                preco_unitario_item REAL NOT NULL,
                PRIMARY KEY (id_pedido_fk, id_produto_fk),
                FOREIGN KEY (id_pedido_fk) REFERENCES Pedidos(id_pedido),
                FOREIGN KEY (id_produto_fk) REFERENCES Produto(id_produto)
            );
        """)
        print("Tabela 'Contem' (itens do pedido) criada com sucesso.")

        # --- Inserindo Dados de Exemplo ---
        # Clientes
        cursor.execute("INSERT INTO Cliente (nome, email, telefone, endereco) VALUES (?, ?, ?, ?)",
                       ("Maria Silva", "maria.silva@email.com", "11987654321", "Rua A, 123"))
        cliente_maria_id = cursor.lastrowid
        cursor.execute("INSERT INTO Cliente (nome, email, telefone, endereco) VALUES (?, ?, ?, ?)",
                       ("João Souza", "joao.souza@email.com", "21998765432", "Av. B, 456"))
        cliente_joao_id = cursor.lastrowid
        print(f"Clientes inseridos: Maria (ID: {cliente_maria_id}), João (ID: {cliente_joao_id})")

        # Vendedores
        cursor.execute("INSERT INTO Vendedor (nome, cpf, comissao) VALUES (?, ?, ?)",
                       ("Carlos Pereira", "12345678901", 0.05))
        vendedor_carlos_id = cursor.lastrowid
        cursor.execute("INSERT INTO Vendedor (nome, cpf, comissao) VALUES (?, ?, ?)",
                       ("Ana Costa", "98765432109", 0.07))
        vendedor_ana_id = cursor.lastrowid
        print(f"Vendedores inseridos: Carlos (ID: {vendedor_carlos_id}), Ana (ID: {vendedor_ana_id})")

        # Produtos
        cursor.execute("INSERT INTO Produto (nome, descricao, preco, estoque) VALUES (?, ?, ?, ?)",
                       ("Notebook XPTO", "Notebook de alta performance", 3500.00, 10))
        produto_notebook_id = cursor.lastrowid
        cursor.execute("INSERT INTO Produto (nome, descricao, preco, estoque) VALUES (?, ?, ?, ?)",
                       ("Mouse Óptico", "Mouse ergonômico sem fio", 75.00, 50))
        produto_mouse_id = cursor.lastrowid
        cursor.execute("INSERT INTO Produto (nome, descricao, preco, estoque) VALUES (?, ?, ?, ?)",
                       ("Teclado Mecânico", "Teclado com switches mecânicos", 250.00, 20))
        produto_teclado_id = cursor.lastrowid
        print(f"Produtos inseridos: Notebook (ID: {produto_notebook_id}), Mouse (ID: {produto_mouse_id}), Teclado (ID: {produto_teclado_id})")

        # Pedidos
        # Pedido 1 para Maria, por Carlos
        data_pedido1 = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        cursor.execute("INSERT INTO Pedidos (data_pedido, valor_total, id_cliente_fk, id_vendedor_fk, status_pedido) VALUES (?, ?, ?, ?, ?)",
                       (data_pedido1, 0.00, cliente_maria_id, vendedor_carlos_id, "Pendente"))
        pedido1_id = cursor.lastrowid

        # Itens do Pedido 1: Notebook (1 und) e Mouse (2 und)
        cursor.execute("INSERT INTO Contem (id_pedido_fk, id_produto_fk, quantidade, preco_unitario_item) VALUES (?, ?, ?, ?)",
                       (pedido1_id, produto_notebook_id, 1, 3500.00))
        cursor.execute("INSERT INTO Contem (id_pedido_fk, id_produto_fk, quantidade, preco_unitario_item) VALUES (?, ?, ?, ?)",
                       (pedido1_id, produto_mouse_id, 2, 75.00))
        valor_total_pedido1 = (1 * 3500.00) + (2 * 75.00)
        cursor.execute("UPDATE Pedidos SET valor_total = ? WHERE id_pedido = ?", (valor_total_pedido1, pedido1_id))
        print(f"Pedido 1 (ID: {pedido1_id}) para Maria (valor: {valor_total_pedido1:.2f}) criado com itens.")

        # Pedido 2 para João, por Ana
        data_pedido2 = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        cursor.execute("INSERT INTO Pedidos (data_pedido, valor_total, id_cliente_fk, id_vendedor_fk, status_pedido) VALUES (?, ?, ?, ?, ?)",
                       (data_pedido2, 0.00, cliente_joao_id, vendedor_ana_id, "Concluído"))
        pedido2_id = cursor.lastrowid

        # Itens do Pedido 2: Teclado Mecânico (1 und)
        cursor.execute("INSERT INTO Contem (id_pedido_fk, id_produto_fk, quantidade, preco_unitario_item) VALUES (?, ?, ?, ?)",
                       (pedido2_id, produto_teclado_id, 1, 250.00))
        valor_total_pedido2 = (1 * 250.00)
        cursor.execute("UPDATE Pedidos SET valor_total = ? WHERE id_pedido = ?", (valor_total_pedido2, pedido2_id))
        print(f"Pedido 2 (ID: {pedido2_id}) para João (valor: {valor_total_pedido2:.2f}) criado com itens.")

        # Comitar as mudanças no banco
        conn.commit()
        print("\nDados de exemplo inseridos com sucesso.")

        # --- Exemplo de Consulta para demonstrar relacionamentos ---
        print("\n--- Relatório de Pedidos Detalhados ---")

        # Consulta SQL para pegar os pedidos detalhados
        cursor.execute("""
            SELECT
                P.id_pedido,
                P.data_pedido,
                C.nome AS nome_cliente,
                V.nome AS nome_vendedor,
                PR.nome AS nome_produto,
                CT.quantidade,
                CT.preco_unitario_item,
                (CT.quantidade * CT.preco_unitario_item) AS subtotal_item
            FROM
                Pedidos P
            JOIN
                Cliente C ON P.id_cliente_fk = C.id_cliente
            JOIN
                Vendedor V ON P.id_vendedor_fk = V.id_vendedor
            JOIN
                Contem CT ON P.id_pedido = CT.id_pedido_fk
            JOIN
                Produto PR ON CT.id_produto_fk = PR.id_produto
            ORDER BY
                P.id_pedido, PR.nome;
        """)

        # Pegando todos os resultados da consulta
        pedidos_detalhados = cursor.fetchall()

        # Exibindo os resultados
        for pedido in pedidos_detalhados:
            print(f"Pedido ID: {pedido['id_pedido']}, Data: {pedido['data_pedido']}, Cliente: {pedido['nome_cliente']}, "
                  f"Vendedor: {pedido['nome_vendedor']}, Produto: {pedido['nome_produto']}, "
                  f"Quantidade: {pedido['quantidade']}, Preço Unitário: {pedido['preco_unitario_item']:.2f}, "
                  f"Subtotal: {pedido['subtotal_item']:.2f}")

    except sqlite3.Error as e:
        print(f"Erro no banco de dados: {e}")
    finally:
        if conn:
            conn.close()

# Chamando a função para criar e preencher o banco de dados
criar_e_preencher_banco_de_dados()


Tabela 'Cliente' criada com sucesso.
Tabela 'Vendedor' criada com sucesso.
Tabela 'Produto' criada com sucesso.
Tabela 'Pedidos' criada com sucesso.
Tabela 'Contem' (itens do pedido) criada com sucesso.
Erro no banco de dados: UNIQUE constraint failed: Cliente.email
